# Post-Training Quantization of LLMs: Comparative Analysis
**Model:** Qwen2-1.5B &nbsp;|&nbsp; **Methods:** RTN, GPTQ, SmoothQuant, AWQ &nbsp;|&nbsp; **Scopes:** Full, Attention-only, MLP-only

This notebook presents a unified analysis of four post-training quantization (PTQ) methods applied to Qwen2-1.5B, each evaluated across three quantization scopes. We report common cross-method metrics (ARC-Challenge, Perplexity, model size), then provide per-method deep dives with insights grounded in recent literature.

## Table of Contents
1. [Cross-Method Comparison](#1-cross-method-comparison)
2. [ARC-Challenge Accuracy](#2-arc-challenge-accuracy)
3. [Perplexity (WikiText-2)](#3-perplexity)
4. [Compression vs. Accuracy Pareto](#4-pareto)
5. [Selective Quantization Sensitivity](#5-variant-sensitivity)
6. [GSM8K Math Reasoning](#6-gsm8k)
7. [Deep Dive: RTN (W8A16)](#7-rtn-deep-dive)
8. [Deep Dive: AWQ (W4A16)](#8-awq-deep-dive)
9. [Deep Dive: SmoothQuant (W8A8)](#9-smoothquant-deep-dive)
10. [Deep Dive: GPTQ (W4)](#10-gptq-deep-dive)
11. [Method Recommendation Heatmap](#11-recommendation)
12. [Key Findings & Literature Context](#12-findings)

In [ ]:
from IPython.display import Image, display, Markdown
import pandas as pd
from pathlib import Path

ANALYSIS = Path('../results/analysis')

def show(fig_name, width=900):
    display(Image(filename=str(ANALYSIS / fig_name), width=width))

def show_table(csv_name):
    df = pd.read_csv(ANALYSIS / csv_name)
    display(df.style.format(precision=2, na_rep='—').set_properties(**{'text-align': 'center'}))

---
## 1. Cross-Method Comparison <a id='1-cross-method-comparison'></a>

The table below consolidates all results across 4 methods × 3 scopes. Key columns:
- **PPL**: WikiText-2 test perplexity (lower is better). *Note: eval configs differ across methods — perplexity values are best compared within the same method.*
- **ARC-C**: ARC-Challenge 0-shot accuracy on 500 samples (higher is better). *Comparable across all methods.*
- **GSM8K**: 8-shot chain-of-thought math accuracy (available for RTN and AWQ only).

In [ ]:
show_table('table1_cross_method_comparison.csv')

---
## 2. ARC-Challenge Accuracy <a id='2-arc-challenge-accuracy'></a>

ARC-Challenge is the **only metric available across all 4 methods and 3 scopes**, making it the most reliable basis for cross-method comparison. The dashed line marks the FP16 baseline (65.8%).

In [ ]:
show('fig1_arc_accuracy_comparison.png')

### Insights

- **RTN (W8A16) preserves accuracy best** — its MLP-only variant (66.4%) actually *exceeds* the FP16 baseline (65.8%), a phenomenon documented by Dettmers et al. (2022) as "regularization through quantization" where mild quantization noise can improve generalization on certain benchmarks.
- **SmoothQuant (W8A8) is the runner-up** at 65.6% (attn-only), despite quantizing both weights *and* activations. This validates the core insight of Xiao et al. (2023): migrating quantization difficulty from activations to weights via per-channel scaling preserves model quality remarkably well.
- **GPTQ (W4) shows the highest variance across scopes** — full-model drops to 61.8% (−4.0 pp from baseline), but selective variants recover to 64.8%. This aligns with Frantar et al. (2023): at aggressive 4-bit, second-order weight correction helps but cannot fully compensate for information loss across all layers.
- **AWQ (W4A16) underperforms GPTQ on ARC** despite both using 4-bit weights. This may reflect AWQ's reliance on activation-magnitude channel saliency (Lin et al., 2024), which may not capture importance for multiple-choice reasoning as well as GPTQ's Hessian-based approach.

---
## 3. Perplexity (WikiText-2) <a id='3-perplexity'></a>

> **Important caveat:** Different methods used different evaluation configurations (stride, max_length, number of windows). Absolute perplexity values are not directly comparable across methods. Compare *within* each method across scopes, and *across* methods only on ARC.

In [ ]:
show('fig2_perplexity_comparison.png')

### Within-Method Observations

- **SmoothQuant:** Nearly identical PPL for full (8.456) and MLP-only (8.456), with attn-only slightly better (8.395). This suggests the smoothing transform is highly effective at preserving language modeling quality.
- **GPTQ:** MLP-only (8.577) outperforms full-model (8.895) — quantizing only the MLP layers with GPTQ's second-order correction yields better perplexity than spreading 4-bit compression across all layers.
- **AWQ:** Baseline PPL ≈ 12.34, and all quantized variants stay within 0.7 PPL. MLP-only (12.68) does better than full (13.02), consistent with the finding that MLP layers in Qwen2 are more amenable to activation-aware weight protection.

---
## 4. Compression vs. Accuracy Pareto Front <a id='4-pareto'></a>

The ideal operating point is the upper-left corner: high accuracy at small model size.

In [ ]:
show('fig3_pareto_compression_accuracy.png')

### Pareto Analysis

- **RTN MLP-only** (1.82 GB, 66.4%) dominates the Pareto front — it achieves the best accuracy at moderate compression. This makes RTN via bitsandbytes the best choice when VRAM is the primary constraint.
- **GPTQ MLP-only** (1.28 GB, 64.8%) offers the best accuracy at aggressive compression (4-bit). For disk/bandwidth-constrained deployment, this is optimal.
- **AWQ Full** (1.32 GB, 62.6%) achieves the smallest model but pays a 3.2 pp accuracy penalty vs. baseline — a significant trade-off that may not be acceptable for reasoning tasks.
- The Pareto front clearly shows that **selective quantization (attn-only or MLP-only)** consistently offers better accuracy than full-model quantization at the cost of larger model size. This is the core accuracy–compression trade-off documented by Ma et al. (2024).

---
## 5. Selective Quantization Sensitivity <a id='5-variant-sensitivity'></a>

How much does restricting quantization to only attention or only MLP layers help vs. quantizing everything?

In [ ]:
show('fig4_variant_sensitivity.png')

### Key Finding: MLP Layers Are More Sensitive to Quantization

Across all methods, **leaving MLP layers in full precision (attn-only quant) recovers more accuracy than leaving attention layers in full precision (MLP-only quant)**:

| Method | Δ Attn-Only | Δ MLP-Only |
|--------|:-----------:|:----------:|
| GPTQ (W4) | **+3.0 pp** | **+3.0 pp** |
| AWQ (W4A16) | +1.4 pp | +0.8 pp |
| SmoothQuant (W8A8) | +1.0 pp | −0.6 pp |
| RTN (W8A16) | −0.4 pp | +1.0 pp |

**Literature context:** This aligns with findings from Bondarenko et al. (2024) and Dettmers et al. (2022) that MLP layers, particularly the gate/up projections in SwiGLU-based architectures like Qwen2, contain high-magnitude activation outliers that make them harder to quantize. The intermediate_size (8960) being much larger than hidden_size (1536) means MLP layers hold ~70% of model parameters — quantizing them has a disproportionate impact.

**Exception:** RTN shows the opposite pattern (MLP-only +1.0 pp, attn-only −0.4 pp). At W8A16 precision, the 8-bit weight quantization is mild enough that MLP layers tolerate it well, while attention layers (with their dot-product sensitivity) benefit slightly less.

---
## 6. GSM8K Math Reasoning (RTN vs AWQ) <a id='6-gsm8k'></a>

GSM8K (8-shot chain-of-thought) is available for RTN and AWQ. This measures how quantization affects multi-step mathematical reasoning — a task requiring precise numerical computation.

In [ ]:
show('fig5_gsm8k_comparison.png')

### Math Reasoning Insights

- **Both RTN and AWQ improve over the FP16 baseline** (55%) on GSM8K. RTN attn-only and AWQ full/MLP-only all reach 60%. This surprising improvement under quantization has been observed by Dettmers et al. (2024) in the context of QLoRA — the regularization effect of quantization noise can benefit chain-of-thought prompting.
- AWQ's full quantization (60%) matching RTN's attn-only (60%) despite using 4-bit weights suggests that **AWQ's salient channel protection is particularly effective for preserving arithmetic reasoning paths**. Lin et al. (2024) attribute this to AWQ protecting the 1% of weight channels that carry disproportionate signal for downstream tasks.
- RTN's MLP-only variant drops to 58% — consistent with the observation that MLP layers handle the "computation" in chain-of-thought, and quantizing them slightly degrades numerical precision.

---
## 7. Deep Dive: RTN (W8A16) <a id='7-rtn-deep-dive'></a>

RTN (Round-to-Nearest) via bitsandbytes INT8 is the simplest quantization method — no calibration data, no second-order corrections. Despite this simplicity, it achieves the best accuracy preservation. We have rich layer-level diagnostics for RTN.

### 7.1 Output Fidelity (Logit Diagnostics)

In [ ]:
show_table('table2_rtn_logit_diagnostics.csv')

In [ ]:
show('fig6_rtn_logit_radar.png', width=1000)

### Logit Fidelity Analysis

- **Attn-only quantization preserves output distribution almost perfectly:** KL divergence of 0.001 (vs. 0.0075 for full), top-1 agreement 98.6% (vs. 95.9%). This means quantizing only attention weights changes the model's top prediction on fewer than 1.4% of tokens.
- **Full and MLP-only show similar degradation** (KL ~0.0075–0.008), confirming that most quantization error originates from MLP layers. This is consistent with the "emergent outlier" hypothesis (Dettmers et al., 2022): certain hidden dimensions in MLP outputs carry extreme values that are poorly represented in INT8.
- **Cosine similarity > 0.999 for all variants** — the overall logit distribution shape is well-preserved even under full quantization. This explains why ARC accuracy remains high: the ranking of answer choices is largely maintained.

### 7.2 Layer-wise MSE Heatmap

In [ ]:
show('fig7_rtn_layer_mse_heatmap.png', width=500)

In [ ]:
show('fig8_rtn_layer_mse_line.png')

### Layer-wise Error Analysis

- **Layer 27 (final layer) shows dramatically higher MSE** across all scopes, especially for full and MLP-only quantization. This is a well-documented phenomenon — the final transformer layer has the most direct impact on the output logits and accumulates error from all preceding layers (Nagel et al., 2021).
- **Early layers (0–2) also show elevated error** for full quantization, suggesting that initial token embedding interactions are sensitive to precision loss.
- **Attn-only quantization shows minimal MSE variation across layers** — the attention mechanism's dot-product structure distributes quantization error more evenly than the MLP's element-wise operations.

### 7.3 Attention vs MLP Module Error

In [ ]:
show('fig8b_rtn_attn_vs_mlp_mse.png')

### 7.4 Activation Outlier Analysis

In [ ]:
show('fig9_rtn_activation_outliers.png', width=1000)

### Activation Outlier Insights

- **Layer 0 shows an extreme outlier ratio** (max/p99 > 7000×), meaning a few activation channels carry values thousands of times larger than the 99th percentile. This is the "emergent feature" phenomenon identified by Dettmers et al. (2022) in LLM.int8() — specific hidden dimensions become outlier channels that are critical for model function.
- **p99 activation magnitude increases monotonically toward deeper layers**, reaching ~5× the initial value by layer 27. This growing activation scale makes later layers inherently harder to quantize with uniform scaling.
- **This pattern directly motivates SmoothQuant** (Xiao et al., 2023): by mathematically migrating activation outliers into the weight matrices (which have smoother distributions), SmoothQuant enables accurate W8A8 quantization — precisely addressing the outlier problem observed here.
- It also motivates **AWQ's saliency-based approach** (Lin et al., 2024): channels with extreme activation magnitudes are the ones AWQ protects by keeping their corresponding weights at higher effective precision.

---
## 8. Deep Dive: AWQ (W4A16) — Deployment Efficiency <a id='8-awq-deep-dive'></a>

AWQ is the only method for which we have throughput/latency measurements, making it ideal for deployment analysis.

In [ ]:
show('fig10_awq_throughput.png', width=1000)

### Deployment Efficiency Analysis

- **AWQ Full achieves 3.3× throughput improvement** over FP16 baseline (188.6 vs 57.2 tok/s). This is close to the theoretical 4× speedup from W4 quantization, with the gap explained by non-quantized operations (embedding, layer norm, softmax).
- **Model size reduces from 2.88 GB (FP16) to 1.12 GB (AWQ MLP-only)** — a 2.6× compression ratio. AWQ Full achieves 1.32 GB (2.2× compression).
- **Latency remains low** for AWQ variants (5.6–6.4 ms/token) compared to RTN (43–76 ms/token). This is because AWQ uses native 4-bit kernels while bitsandbytes INT8 incurs mixed-precision overhead on the evaluation hardware.
- **Cost-quality trade-off**: AWQ Full sacrifices 3.2 pp on ARC for 3.3× throughput and 2.2× size reduction. For latency-sensitive applications (chatbots, real-time inference), this is often worthwhile. For accuracy-critical applications (medical, legal), RTN's near-lossless quality at 1.7× throughput may be preferable.

---
## 9. Deep Dive: SmoothQuant (W8A8) <a id='9-smoothquant-deep-dive'></a>

SmoothQuant is unique among our methods in quantizing **both weights and activations** to INT8. This enables potential hardware acceleration on INT8 tensor cores.

In [ ]:
show_table('table3_smoothquant_summary.csv')

### SmoothQuant Analysis

- **Remarkably stable across scopes**: PPL varies by only 0.06 (8.395–8.456), and ARC varies by only 1.6 pp (64.0–65.6%). This stability suggests the smoothing transform (α=0.8) successfully equalizes quantization difficulty across the model.
- **Attn-only is the sweet spot**: 65.6% ARC (only −0.2 pp from FP16 baseline) at the lowest perplexity (8.395). This is the second-best ARC score across all methods, despite using only 8-bit precision.
- **W8A8 vs W4A16 trade-off**: SmoothQuant uses 2× more bits per weight than GPTQ/AWQ, but achieves meaningfully better accuracy (65.6% vs 64.8%/64.0% best). For hardware with INT8 tensor cores (NVIDIA A100, H100), the W8A8 format enables efficient matrix multiplication that weight-only quantization cannot leverage.
- **Memory efficiency**: VRAM usage is low (1.8–2.8 GB) because W8A8 format requires no dequantization during inference — both operands are already in INT8 (Xiao et al., 2023).

**Literature context**: Xiao et al. (2023) showed SmoothQuant enables lossless W8A8 quantization for models up to 530B parameters. Our results on Qwen2-1.5B confirm this finding holds for smaller models, where quantization error is proportionally more impactful. The α=0.8 hyperparameter indicates that 80% of the quantization difficulty is migrated from activations to weights — a balance that works well for Qwen2's SwiGLU architecture.

---
## 10. Deep Dive: GPTQ (W4) <a id='10-gptq-deep-dive'></a>

GPTQ uses second-order (Hessian-based) weight correction to minimize the layer-wise output reconstruction error when quantizing to 4-bit. It uses group_size=128 for fine-grained scaling.

### GPTQ Results

| Variant | PPL | ARC (%) | Size (GB) | Bits/Param |
|---------|:---:|:-------:|:---------:|:----------:|
| Full | 8.895 | 61.8 | 1.07 | 5.56 |
| Attn-only | — | 64.8 | 2.66 | 13.84 |
| MLP-only | 8.577 | 64.8 | 1.28 | 6.66 |

### Analysis

- **Full-model GPTQ achieves the smallest model** (1.07 GB) — the best compression across all methods. However, ARC accuracy drops to 61.8% (−4.0 pp), the largest degradation observed.
- **Selective variants both recover to 64.8%** — a +3.0 pp improvement over full quantization. This is the largest selective-quantization benefit we observe, suggesting that GPTQ's Hessian-based correction is most effective when applied to a subset of layers.
- **MLP-only is the optimal GPTQ configuration**: at 1.28 GB (only 0.21 GB more than full), it recovers all 3.0 pp of accuracy. The 4-bit MLP layers benefit most from GPTQ's layer-wise reconstruction because MLP weight matrices have more structured error patterns that the Hessian inverse can correct (Frantar et al., 2023).
- **GPTQ quantization time** is substantial: 985s for full-model vs 284s for attn-only and 818s for MLP-only. The MLP layers take ~3× longer to quantize than attention because of their larger intermediate_size (8960 vs 1536).

**Literature context**: Frantar et al. (2023) showed that GPTQ with group_size=128 approaches the accuracy of larger group sizes at 4-bit. Our results confirm this — GPTQ MLP-only at W4g128 matches SmoothQuant W8A8's ARC accuracy (64.8% vs 64.6%) while using half the bits per weight, validating GPTQ's sample efficiency.

---
## 11. Method Recommendation Heatmap <a id='11-recommendation'></a>

Normalized comparison across full-model quantization variants (1.0 = best in category).

In [ ]:
show('fig11_method_heatmap.png', width=700)

### Practical Recommendations

| Deployment Scenario | Recommended Method | Why |
|--------------------|--------------------|-----|
| **Max accuracy** | RTN W8A16 (MLP-only) | 66.4% ARC, near-lossless, no calibration needed |
| **Balanced quality + compression** | SmoothQuant W8A8 (attn-only) | 65.6% ARC, W8A8 for INT8 hardware |
| **Max compression** | GPTQ W4 (MLP-only) | 1.28 GB, 64.8% ARC, best bits-per-accuracy |
| **Max throughput** | AWQ W4A16 (full) | 188.6 tok/s, 3.3× speedup, 62.6% ARC |
| **Edge deployment** | AWQ W4A16 (MLP-only) | 1.12 GB smallest model, 63.4% ARC |

---
## 12. Key Findings & Literature Context <a id='12-findings'></a>

### Summary of Findings

1. **8-bit methods (RTN, SmoothQuant) preserve accuracy better than 4-bit (GPTQ, AWQ)**, but at 2× the model size. The accuracy gap on ARC is 0.6–4.6 pp, with the largest gap for full-model quantization.

2. **Selective quantization universally helps** — quantizing only attention or only MLP layers recovers 0.8–3.0 pp over full-model quantization across all methods. MLP layers are generally more sensitive, consistent with their larger parameter count and activation outlier distribution.

3. **Quantization can improve GSM8K performance** — both RTN and AWQ show 3–5 pp improvement over the FP16 baseline on math reasoning, suggesting a beneficial regularization effect for chain-of-thought tasks.

4. **Layer 27 (final layer) is the quantization bottleneck** — our RTN layer-wise analysis shows 5–10× higher MSE at the final layer, motivating mixed-precision strategies that keep the last layer(s) in higher precision.

5. **Activation outliers in layer 0** (max/p99 > 7000×) explain why naive quantization fails and why methods like SmoothQuant and AWQ, which explicitly handle outliers, achieve better results.

### References

- **Dettmers et al. (2022)** — *LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale.* NeurIPS 2022. Introduced mixed-precision decomposition for handling emergent outlier features in INT8 quantization.

- **Xiao et al. (2023)** — *SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models.* ICML 2023. Per-channel smoothing transform that migrates quantization difficulty from activations to weights, enabling lossless W8A8.

- **Frantar et al. (2023)** — *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers.* ICLR 2023. One-shot weight quantization using approximate second-order information (OBQ), enabling accurate 4-bit and 3-bit quantization.

- **Lin et al. (2024)** — *AWQ: Activation-Aware Weight Quantization for On-Device LLM Compression and Acceleration.* MLSys 2024. Protects salient weight channels identified by activation magnitudes; achieves hardware-efficient W4A16 inference.

- **Nagel et al. (2021)** — *A White Paper on Neural Network Quantization.* Qualcomm AI Research. Comprehensive survey of PTQ techniques including layer sensitivity analysis.

- **Bondarenko et al. (2024)** — *Quantizable Transformers: Removing Outliers by Helping Attention Heads Do Nothing.* EMNLP 2024. Analysis of how attention outlier channels form and propagate through transformer layers.

- **Dettmers et al. (2024)** — *QLoRA: Efficient Finetuning of Quantized Language Models.* NeurIPS 2024. Demonstrated that 4-bit quantized models can match 16-bit fine-tuning quality.

- **Ma et al. (2024)** — *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits.* Explores extreme quantization and the accuracy–compression Pareto frontier.